# Ligne **Léa Petit** dans `database_etudiants_300k.csv`

Exécuter les cellules depuis le dossier `Site/moc_data/scrypte` (ou adapter `CSV_PATH`). Lecture par chunks pour éviter de charger 300k lignes inutilement si la ligne est trouvée tôt.

In [1]:
from pathlib import Path

import pandas as pd

# Chemin vers le CSV (dossier courant = scrypte/)
CSV_PATH = Path("../Résultats/database_etudiants_300k.csv").resolve()
if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV introuvable: {CSV_PATH} — lance le notebook depuis Site/moc_data/scrypte")

CSV_PATH

PosixPath("/Users/fefe/Desktop/Cours M1 Albert/Semestre 2/Alberthon/L'Etudiant/Site/moc_data/Résultats/database_etudiants_300k.csv")

In [2]:
def mask_lea_petit(df: pd.DataFrame) -> pd.Series:
    """Filtre Léa / Lea + Petit, ou email contenant petit + lea / léa (sans regex sur le point)."""
    ln = df["nom"].astype(str).str.strip().str.lower()
    lp = df["prenom"].astype(str).str.strip().str.lower()
    by_name = (ln == "petit") & lp.isin(["léa", "lea"])
    em = df["email"].astype(str).str.lower()
    by_email = em.str.contains("petit", na=False, regex=False) & (
        em.str.contains("lea.", na=False, regex=False)
        | em.str.contains("léa.", na=False, regex=False)
    )
    return by_name | by_email


def load_lea_petit_rows(csv_path: Path, chunksize: int = 100_000) -> pd.DataFrame:
    hits: list[pd.DataFrame] = []
    for chunk in pd.read_csv(csv_path, chunksize=chunksize, low_memory=False):
        m = mask_lea_petit(chunk)
        if m.any():
            hits.append(chunk.loc[m].copy())
    if not hits:
        return pd.DataFrame()
    return pd.concat(hits, ignore_index=True)


df_lea = load_lea_petit_rows(CSV_PATH)
df_lea

,id,date_inscription,prenom,nom,niveau_actuel,ville,source_lead,ecole_actuelle,email,tel,...,M2_s1_majeure,M2_s1_anglais,M2_s1_projets,M2_s1_option,M2_s1_moyenne_G,M2_s2_majeure,M2_s2_anglais,M2_s2_projets,M2_s2_option,M2_s2_moyenne_G
0,109,2025-12-30,Léa,Petit,Bac+3,Bordeaux,Recherche Google,Université Paris-Saclay,léa.petit109@mail.fr,649208212,...,12.79,15.07,15.42,14.48,18.85,13.71,11.47,19.57,14.43,15.50
1,117,2025-07-03,Léa,Petit,2nde,Lyon,QR Code Lycée,INSA,léa.petit117@mail.fr,685809237,...,8.59,9.17,10.44,12.13,7.41,14.95,9.42,11.49,12.23,8.06
2,311,2025-10-31,Léa,Petit,2nde,Nantes,QR Code Lycée,Epitech,léa.petit311@mail.fr,638096538,...,18.41,15.18,11.83,16.07,9.05,15.64,19.44,14.07,15.66,17.30
3,410,2025-09-01,Léa,Petit,Bac+3,Montpellier,Instagram Ad,Université Paris-Saclay,léa.petit410@mail.fr,676986733,...,10.97,7.43,15.06,10.09,14.39,7.79,11.25,9.24,14.90,9.67
4,415,2025-02-22,Léa,Petit,Bac+1,Nantes,Instagram Ad,Université Paris-Saclay,léa.petit415@mail.fr,657440428,...,11.37,9.46,8.62,10.41,11.83,11.89,12.50,8.90,7.95,7.61
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3063,299557,2025-11-08,Léa,Petit,Terminale,Bordeaux,QR Code Lycée,Lycée Condorcet,léa.petit299557@mail.fr,673275080,...,10.94,15.94,14.91,16.06,18.49,14.87,16.67,12.41,16.27,15.91
3064,299648,2026-02-18,Léa,Petit,Terminale,Nantes,Instagram Ad,Lycée Henri IV,léa.petit299648@mail.fr,619071945,...,9.76,10.65,8.19,8.00,7.26,4.93,11.68,9.62,8.05,7.03
3065,299760,2025-01-18,Léa,Petit,Bac+3,Montpellier,Instagram Ad,Lycée Condorcet,léa.petit299760@mail.fr,621171857,...,10.88,10.68,8.80,10.87,11.17,8.09,11.79,12.02,8.32,12.97
3066,299820,2026-03-23,Léa,Petit,1ère,Lyon,Recherche Google,HEC,léa.petit299820@mail.fr,680954334,...,9.65,8.68,9.76,9.88,6.75,7.87,10.81,7.21,12.51,5.39


In [ ]:
# Une seule ligne « canon » (id=2) si présente, sinon toutes les correspondances
if df_lea.empty:
    print("Aucune ligne trouvée pour Léa Petit.")
else:
    if (df_lea["id"] == 2).any():
        df_out = df_lea[df_lea["id"] == 2].copy()
    else:
        df_out = df_lea.copy()
    display(df_out)
    print(f"Nombre de lignes: {len(df_out)}")